In [ ]:
# -*- coding: utf-8 -*-
"""main_v3_dual_evaluator.py

LangGraph Joke Generation Pipeline with Dual Evaluation (GPT-OSS + Mistral)
"""

# !pip uninstall -y transformers accelerate peft bitsandbytes trl
# !pip install -q -U bitsandbytes accelerate peft transformers trl datasets
# !pip install faiss-cpu

import os
import json
import torch
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pickle
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss

import wandb
from huggingface_hub import notebook_login

In [ ]:
"""
Setup environment and imports
"""

print("🔧 Setting up environment...")

# Check for GPU
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected. Will use CPU (slower)")

print("✓ Environment setup complete\n")

"""
LangGraph Joke Generation Pipeline - Orchestrator with Dual Evaluation
Imports and orchestrates existing implementations
"""

from typing import TypedDict, Literal, List, Dict
from langgraph.graph import StateGraph, END
import sys
from pathlib import Path

# Add project paths for imports (Windows-compatible)
project_root = Path(__file__).parent.parent.parent if '__file__' in globals() else Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src" / "generation"))
sys.path.insert(0, str(project_root / "src" / "evaluation"))

print(f"📁 Project root: {project_root}")
print(f"📂 Python paths added:")
print(f"   - {project_root / 'src' / 'generation'}")
print(f"   - {project_root / 'src' / 'evaluation'}\n")

# Import your existing implementations
from using_llama_with_rag import RAGHumorGenerator, RAGConfig
from gpt_joke_generation import GPT4JokeGenerator
from evaluation.gpt_oss_evaluator import GPTOSSHumorScorer
from evaluation.mistral_evaluator import MistralHumorScorer

print("✓ All modules imported successfully\n")


In [ ]:

# ============================================================================
# LANGGRAPH STATE DEFINITION
# ============================================================================

class JokeGenerationState(TypedDict):
    """State for the joke generation pipeline"""
    # Input
    word1: str
    word2: str

    # Generation tracking
    llama_jokes: List[str]
    gpt_jokes: List[str]
    candidate_jokes: List[str]

    # Scoring tracking
    scores: List[float]
    gpt_scores: List[float]      # NEW: Track GPT scores separately
    mistral_scores: List[float]  # NEW: Track Mistral scores separately

    # Best result tracking
    best_joke: str
    best_score: float

    # Iteration tracking
    iteration_count: int
    max_iterations: int

    # History for debugging
    history: List[Dict]



In [ ]:

# ============================================================================
# NODE FUNCTIONS
# ============================================================================

def generate_jokes_llama(state: JokeGenerationState, llama_gen: RAGHumorGenerator) -> Dict:
    """Generate jokes using Llama model"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    print(f"\\n[Iteration {iteration + 1}] Generating jokes with Llama ...")

    result_with_RAG = llama_gen.generate_with_rag(
        word1=word1,
        word2=word2,
        use_rag=True,  # with RAG
        num_candidates=5
    )

    result_without_RAG = llama_gen.generate_with_rag(
        word1=word1,
        word2=word2,
        use_rag=False,  # No RAG - just fine-tuned model
        num_candidates=5
    )

    jokes = result_with_RAG["generated_jokes"] + result_without_RAG["generated_jokes"]

    print('result_with_RAG jokes:')
    print(result_with_RAG["generated_jokes"])

    print('result_without_RAG jokes:')
    print(result_without_RAG["generated_jokes"])

    print(f"   Generated {len(jokes)} Llama jokes")

    return {"llama_jokes": jokes}


def generate_jokes_gpt(state: JokeGenerationState, gpt_gen: GPT4JokeGenerator) -> Dict:
    """Generate jokes using GPT-OSS"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    print(f"\\n[Iteration {iteration + 1}] Generating jokes with GPT-OSS...")

    jokes = gpt_gen.generate_jokes(word1, word2, num_jokes=5, verbose=False)

    print(f"   Generated {len(jokes)} GPT-OSS jokes")

    return {"gpt_jokes": jokes}


def combine_jokes(state: JokeGenerationState) -> JokeGenerationState:
    """Combine jokes from both generators"""
    llama_jokes = state.get("llama_jokes", [])
    gpt_jokes = state.get("gpt_jokes", [])

    all_jokes = llama_jokes + gpt_jokes

    print(f"\\n✅ Combined {len(all_jokes)} total jokes ({len(llama_jokes)} Llama + {len(gpt_jokes)} GPT)")

    return {
        **state,
        "candidate_jokes": all_jokes
    }


def score_jokes_dual(state: JokeGenerationState, gpt_scorer: GPTOSSHumorScorer, mistral_scorer: MistralHumorScorer) -> JokeGenerationState:
    """Score all candidate jokes using BOTH GPT-OSS and Mistral scorers, then average"""
    jokes = state["candidate_jokes"]
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    print(f"\\n📊 [Iteration {iteration + 1}] Scoring {len(jokes)} jokes with GPT-OSS + Mistral...")

    # Score with GPT-OSS (range 0-10)
    print("   Scoring with GPT-OSS...")
    gpt_scores = gpt_scorer.score_jokes(jokes, word1, word2, verbose=False)
    
    # Score with Mistral (range 0-4)
    print("   Scoring with Mistral...")
    mistral_scores = mistral_scorer.score_jokes(jokes, word1, word2, verbose=False)
    
    # Normalize Mistral scores to 0-10 scale to match GPT
    mistral_scores_normalized = [score * 2.5 for score in mistral_scores]  # 4 * 2.5 = 10
    
    # Average the two scores
    final_scores = [
        (gpt + mistral) / 2.0
        for gpt, mistral in zip(gpt_scores, mistral_scores_normalized)
    ]

    print(f"\\n   📋 Score breakdown for each joke:")
    print(f"   {'='*100}")
    for i, (joke, gpt, mistral_raw, mistral_norm, final) in enumerate(
        zip(jokes, gpt_scores, mistral_scores, mistral_scores_normalized, final_scores)
    ):
        print(f"   [{i+1:2d}] GPT: {gpt:5.2f}/10 | Mistral: {mistral_raw:4.2f}/4 ({mistral_norm:5.2f}/10) | Final: {final:5.2f}/10")
        print(f"        {joke[:90]}...")
        print()

    # Find best joke
    if final_scores:
        best_idx = final_scores.index(max(final_scores))
        best_joke = jokes[best_idx]
        best_score = final_scores[best_idx]
    else:
        best_joke = ""
        best_score = 0.0

    print(f"\\n   🎯 Best score: {best_score:.2f}/10 (GPT: {gpt_scores[best_idx]:.2f}, Mistral: {mistral_scores[best_idx]:.2f})")
    print(f"   📈 Average score: {sum(final_scores)/len(final_scores):.2f}/10")
    print(f"   📊 Score range: {min(final_scores):.2f} - {max(final_scores):.2f}")

    return {
        **state,
        "scores": final_scores,
        "gpt_scores": gpt_scores,
        "mistral_scores": mistral_scores,
        "best_joke": best_joke,
        "best_score": best_score
    }


def check_score_threshold(state: JokeGenerationState) -> Literal["refine", "output"]:
    """Decision node: check if score meets threshold"""
    score = state["best_score"]
    iteration = state.get("iteration_count", 0)
    max_iter = state.get("max_iterations", 3)

    print(f"\\n🎯 Decision: Score={score:.2f}, Iteration={iteration + 1}/{max_iter}")

    if score >= 8.0:
        print(f"   ✓ Score threshold met! (≥8.0)")
        return "output"
    elif iteration >= max_iter - 1:
        print(f"   ⚠️ Max iterations reached, outputting best result")
        return "output"
    else:
        print(f"   🔄 Score below threshold, refining...")
        return "refine"


def increment_iteration(state: JokeGenerationState) -> JokeGenerationState:
    """Increment iteration counter and save history"""
    current_iter = state.get("iteration_count", 0)
    history = state.get("history", [])

    # Save this iteration's results
    history.append({
        "iteration": current_iter + 1,
        "best_score": state["best_score"],
        "best_joke": state["best_joke"],
        "num_candidates": len(state.get("candidate_jokes", [])),
        "avg_score": sum(state.get("scores", [0])) / max(len(state.get("scores", [1])), 1),
        "avg_gpt_score": sum(state.get("gpt_scores", [0])) / max(len(state.get("gpt_scores", [1])), 1),
        "avg_mistral_score": sum(state.get("mistral_scores", [0])) / max(len(state.get("mistral_scores", [1])), 1)
    })

    return {
        **state,
        "iteration_count": current_iter + 1,
        "history": history
    }


def output_result(state: JokeGenerationState) -> JokeGenerationState:
    """Final output node"""
    print(f"\\n{'='*80}")
    print(f"🎭 FINAL RESULT")
    print(f"{'='*80}")
    print(f"\\nWord Pair: '{state['word1']}' + '{state['word2']}'")
    print(f"Final Score: {state['best_score']:.2f}/10")
    print(f"Total Iterations: {state.get('iteration_count', 0) + 1}")
    print(f"\\nSelected Joke:")
    print(f"  {state['best_joke']}")

    # Show iteration progress
    if state.get("history"):
        print(f"\\n📈 Iteration History:")
        print(f"{'Iter':<6} {'Final':<8} {'GPT':<8} {'Mistral':<9} {'Avg Final':<10} {'Candidates':<12}")
        print(f"{'-'*70}")
        for h in state["history"]:
            print(f"{h['iteration']:<6} {h['best_score']:<8.2f} "
                  f"{h.get('avg_gpt_score', 0):<8.2f} {h.get('avg_mistral_score', 0):<9.2f} "
                  f"{h['avg_score']:<10.2f} {h['num_candidates']:<12}")

    print(f"\\n{'='*80}\\n")

    return state

In [ ]:



# ============================================================================
# GRAPH BUILDER
# ============================================================================

def build_joke_generation_graph(
    llama_generator: RAGHumorGenerator,
    gpt_generator: GPT4JokeGenerator,
    gpt_scorer: GPTOSSHumorScorer,
    mistral_scorer: MistralHumorScorer
):
    """
    Build the LangGraph for joke generation pipeline with dual evaluation.

    Args:
        llama_generator: RAGHumorGenerator
        gpt_generator: GPT4JokeGenerator
        gpt_scorer: GPTOSSHumorScorer
        mistral_scorer: MistralHumorScorer

    Returns:
        Compiled LangGraph
    """
    workflow = StateGraph(JokeGenerationState)

    # Add nodes with bound generators/scorers
    workflow.add_node(
        "generate_llama",
        lambda state: generate_jokes_llama(state, llama_generator)
    )
    workflow.add_node(
        "generate_gpt",
        lambda state: generate_jokes_gpt(state, gpt_generator)
    )
    workflow.add_node("combine_jokes", combine_jokes)
    workflow.add_node(
        "score_jokes",
        lambda state: score_jokes_dual(state, gpt_scorer, mistral_scorer)
    )
    workflow.add_node("increment_iteration", increment_iteration)
    workflow.add_node("output", output_result)

    # Set entry point - start both generators in parallel
    workflow.set_entry_point("generate_llama")
    workflow.set_entry_point("generate_gpt")

    # Add edges - both generators feed into combine
    workflow.add_edge("generate_llama", "combine_jokes")
    workflow.add_edge("generate_gpt", "combine_jokes")

    # After combining, score the jokes
    workflow.add_edge("combine_jokes", "score_jokes")

    # Decision point: refine or output
    workflow.add_conditional_edges(
        "score_jokes",
        check_score_threshold,
        {
            "refine": "increment_iteration",
            "output": "output"
        }
    )

    # Refinement loop - go back to both generators
    workflow.add_edge("increment_iteration", "generate_llama")
    workflow.add_edge("increment_iteration", "generate_gpt")

    # End the workflow
    workflow.add_edge("output", END)

    return workflow.compile()


In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("="*80)
print("LANGGRAPH JOKE GENERATION PIPELINE - DUAL EVALUATOR MODE")
print("GPT-OSS + Mistral Ensemble Scoring")
print("="*80)
print()

# Configuration - Load from environment
from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found in environment. Please set it in .env file")

# RAG Configuration
config = RAGConfig()
config.TEMPERATURE = 0.65
config.TOP_K = 50

# Initialize components
print("🚀 Initializing components...\n")

# 1. Initialize Llama Generator
try:
    llama_gen = RAGHumorGenerator(config)
    llama_gen.load_model()
    llama_gen.load_rag_database(build_if_missing=True)
    print("✓ Llama generator initialized\n")
except Exception as e:
    print(f"⚠️ Could not load Llama model: {e}")
    print("   Continuing without Llama generator...\n")
    llama_gen = None

# 2. Initialize GPT-OSS Generator
gpt_gen = GPT4JokeGenerator(
    api_key=NVIDIA_API_KEY,
    temperature=0.8,
    max_retries=1
)
print("✓ GPT-OSS generator initialized\n")

# 3. Initialize GPT-OSS Scorer
gpt_scorer = GPTOSSHumorScorer(
    api_key=NVIDIA_API_KEY,
    temperature=0.4,
    max_retries=2
)
print("✓ GPT-OSS scorer initialized\n")

# 4. Initialize Mistral Scorer
mistral_checkpoint = project_root / "src" / "evaluation" / "mistral_evaluator" / "checkpoints_ministral3_multitask"
print(f"📁 Looking for Mistral checkpoint at: {mistral_checkpoint}")

if not mistral_checkpoint.exists():
    print(f"⚠️ Mistral checkpoint not found!")
    print(f"   Expected location: {mistral_checkpoint}")
    print(f"   Please train the model first using finetune_mistral3.ipynb")
    print(f"   Or update the checkpoint path below.\n")
    mistral_scorer = None
else:
    mistral_scorer = MistralHumorScorer(
        base_model_id="mistralai/Ministral-3-8B-Base-2512",
        checkpoint_path=str(mistral_checkpoint),
        num_labels=2,
        max_length=512
    )
    mistral_scorer.load_model()
    print("✓ Mistral scorer initialized\n")

# Build the graph (only if we have both scorers)
if mistral_scorer is None:
    print("❌ Cannot run dual evaluator mode without Mistral scorer")
    print("   Please train the Mistral model first using finetune_mistral3.ipynb\n")
    sys.exit(1)

if llama_gen is None:
    print("❌ Cannot run pipeline without Llama generator")
    print("   Please ensure Llama model is available\n")
    sys.exit(1)

graph = build_joke_generation_graph(llama_gen, gpt_gen, gpt_scorer, mistral_scorer)
print("✓ LangGraph compiled (Llama + GPT-OSS + Mistral Dual Evaluator mode)\n")

# Test word pairs
test_pairs = [
    ("banana", "satellite"),
    ("angry", "teacup"),
    ("cosmic", "toaster"),
    ("sleepy", "volcano"),
    ("neon", "hamster"),
    ("awkward", "penguin"),
    ("electric", "spaghetti"),
    ("dramatic", "cactus"),
    ("confused", "astronaut"),
    ("rubber", "tornado"),
    ("invisible", "taco"),
    ("sarcastic", "dolphin"),
    ("haunted", "calculator"),
    ("spicy", "cloud"),
    ("polite", "chainsaw"),
    ("disco", "librarian"),
    ("lonely", "robot"),
    ("caffeinated", "squirrel"),
    ("confused", "mailbox")
]

# Run the pipeline for each word pair
results_summary = []

for word1, word2 in test_pairs:
    print("\n" + "="*80)
    print(f"PROCESSING: '{word1}' + '{word2}'")
    print("="*80)

    # Initial state
    initial_state = {
        "word1": word1,
        "word2": word2,
        "llama_jokes": [],
        "gpt_jokes": [],
        "candidate_jokes": [],
        "scores": [],
        "gpt_scores": [],
        "mistral_scores": [],
        "best_joke": "",
        "best_score": 0.0,
        "iteration_count": 0,
        "max_iterations": 1,
        "history": []
    }

    try:
        # Run the graph
        result = graph.invoke(initial_state)

        # Additional result summary
        print(f"\n{'🎨'*40}")
        print("SUMMARY")
        print(f"{'🎨'*40}")
        total_candidates = sum(h['num_candidates'] for h in result.get('history', [])) + len(result.get('candidate_jokes', []))
        print(f"Total Jokes Generated: {total_candidates}")
        print(f"Success: {'✓' if result['best_score'] >= 8.0 else '✗'} (threshold: 8.0)")
        
        # Store summary
        results_summary.append({
            "word_pair": f"{word1} + {word2}",
            "best_score": result['best_score'],
            "best_joke": result['best_joke'],
            "iterations": result.get('iteration_count', 0) + 1,
            "total_candidates": total_candidates
        })

    except Exception as e:
        print(f"\n❌ Error processing word pair: {e}")
        import traceback
        traceback.print_exc()

# Final summary of all word pairs
print("\n" + "="*80)
print("📊 OVERALL RESULTS SUMMARY")
print("="*80)
print(f"\n{'Word Pair':<30} {'Score':<10} {'Iterations':<12} {'Candidates':<12} {'Success':<8}")
print("-"*80)
for r in results_summary:
    success = "✓" if r['best_score'] >= 8.0 else "✗"
    print(f"{r['word_pair']:<30} {r['best_score']:<10.2f} {r['iterations']:<12} {r['total_candidates']:<12} {success:<8}")

avg_score = sum(r['best_score'] for r in results_summary) / len(results_summary) if results_summary else 0
success_rate = sum(1 for r in results_summary if r['best_score'] >= 8.0) / len(results_summary) * 100 if results_summary else 0

print("-"*80)
print(f"Average Score: {avg_score:.2f}/10")
print(f"Success Rate: {success_rate:.1f}% (score ≥ 8.0)")

print("\n" + "="*80)
print("✅ Pipeline execution complete!")
print("="*80)
